[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/pml-f2026-notebooks/blob/main/class-demos/session04_numpy_pandas.ipynb)

# Session 4 deck code, assembled in slide order

**Session 4 · the in-class demo — runs on the free tier of Colab, nothing to install**

Generated by scripts/make_demo.py from the slides themselves, so it stays
honest about what the deck actually shows. Run it before class.

Each heading names the slide its cell accompanies; the outputs below were saved from a real run.

## Slide 3: The Data, and a Condition

In [1]:
import numpy as np

# six houses: area (sq ft), bedrooms, price ($1000s)
area = np.array([1400, 1800, 2400, 1100, 3000, 1650])
beds = np.array([3, 3, 4, 2, 5, 3])
price = np.array([245., 312., 379.,
                  189., 505., 279.])
mask = price < 300      # elementwise comparison
print(mask)
print(mask.sum(), mask.mean())

[ True False False  True False  True]
3 0.5


## Slide 5: Selecting Rows With a Mask

In [2]:
homes = np.column_stack([area, beds, price])
print(homes.shape, homes.dtype)
print(homes[mask])   # a 1-D mask picks whole rows

(6, 3) float64
[[1400.    3.  245.]
 [1100.    2.  189.]
 [1650.    3.  279.]]


## Slide 6: Combining Conditions

In [3]:
big_cheap = (area > 1500) & (price < 320)
print(big_cheap)
print(homes[big_cheap])
print(price[~(beds >= 4)])

[False  True False False False  True]
[[1800.    3.  312.]
 [1650.    3.  279.]]
[245. 312. 189. 279.]


## Slide 7: Fancy Indexing

In [4]:
order = [3, 0, 5]        # positions, in this order
print(price[order])
print(homes[order])
pick = np.array([[0, 1], [3, 5]])   # a 2-D index
print(price[pick])

[189. 245. 279.]
[[1100.    2.  189.]
 [1400.    3.  245.]
 [1650.    3.  279.]]
[[245. 312.]
 [189. 279.]]


## Slide 8: Views, Copies, and the Consequence

In [5]:
a = np.arange(6)
view = a[1:4]       # a basic slice is a VIEW
view[0] = 99
print(a)            # a changed
b = np.arange(6)
picked = b[b > 2]   # boolean index -> a COPY
picked[0] = 99
print(b)            # b did not change
print(view.base is a, picked.base is b)

[ 0 99  2  3  4  5]
[0 1 2 3 4 5]
True False


## Slide 9: Worked Question: Where Did the Write Go?

In [6]:
sale = homes.copy()               # scratch copy
cheap = sale[price < 300];  cheap[:, 2] = 0
print(sale[0, 2])
first = sale[:2];           first[:, 2] = 0
print(sale[0, 2])

245.0
0.0


## Slide 10: np.where and np.select

In [7]:
tier = np.where(price < 300, "budget", "premium")
print(tier)

bands = np.select(
    [price < 250, price < 400],   # conditions
    ["low", "mid"],               # matching results
    default="high")
print(bands)

['budget' 'premium' 'premium' 'budget' 'premium' 'budget']
['low' 'mid' 'mid' 'low' 'high' 'mid']


## Slide 11: Sorting and argsort

In [8]:
print(np.sort(price))      # the values, ascending
idx = np.argsort(price)
print(idx)                 # the positions that sort
print(homes[np.argsort(price)[::-1]][:3])

[189. 245. 279. 312. 379. 505.]
[3 0 5 1 2 4]
[[3000.    5.  505.]
 [2400.    4.  379.]
 [1800.    3.  312.]]


## Slide 12: Matrix Multiplication: @

In [9]:
w = np.array([0.1, 20.0])   # $/sq ft, $1000s per bed
X = np.column_stack([area, beds])
print(X.shape, w.shape)
print(X @ w)                # one number per house

(6, 2) (2,)
[200. 240. 320. 150. 400. 225.]


## Slide 13: The Normal Equations

In [10]:
ones = np.ones(len(area))
A = np.column_stack([ones, area, beds])
print(A.shape)
# solve (A'A) coef = A'price  for coef
coef = np.linalg.solve(A.T @ A, A.T @ price)
print(np.round(coef, 4))

(6, 3)
[ 9.1392  0.1413 12.5371]


## Slide 14: lstsq: The Safer Default

In [11]:
coef, *_ = np.linalg.lstsq(A, price, rcond=None)
print(np.round(coef, 4))

dup = area * 1.0000001     # the same column twice
A_bad = np.column_stack([ones, area, dup])
print(f"{np.linalg.cond(A_bad):.3e}")

[ 9.1392  0.1413 12.5371]
2.282e+16


## Slide 16: Random Numbers: default_rng

In [12]:
rng = np.random.default_rng(0)
print(np.round(rng.normal(size=4), 4))
print(rng.integers(0, 10, size=5))
print(np.round(rng.uniform(size=3), 4))

[ 0.1257 -0.1321  0.6404  0.1049]
[1 8 6 9 5]
[0.7295 0.5436 0.9351]


## Slide 17: Seeding and Reproducibility

In [13]:
rng1 = np.random.default_rng(0)
rng2 = np.random.default_rng(0)
print(np.allclose(rng1.normal(size=100),
                  rng2.normal(size=100)))

rng3 = np.random.default_rng(0)
print(np.round(rng3.normal(size=3), 4))
print(np.round(rng3.normal(size=3), 4))

True
[ 0.1257 -0.1321  0.6404]
[ 0.1049 -0.5357  0.3616]


## Slide 18: Check Yourself #2

In [14]:
rng = np.random.default_rng(0)
a = rng.integers(0, 10, size=3)
b = rng.integers(0, 10, size=3)
rng2 = np.random.default_rng(0)
c = rng2.integers(0, 10, size=3)
print(a, b, c)

[8 6 5] [2 3 0] [8 6 5]


## Slide 19: Where ndarray Runs Out

In [15]:
city = np.array(["Moscow", "Boise", "Boise",
                 "Moscow", "Boise", "Moscow"])
mixed = np.column_stack([city, area, price])
print(mixed.dtype)
print(mixed[0])
print(np.array([1.0, np.nan, 3.0]).mean())

<U32
['Moscow' '1400' '245.0']
nan


## Slide 20: Series: A Labeled 1-D Array

In [16]:
import pandas as pd

idx = ["h1", "h2", "h3", "h4", "h5", "h6"]
s = pd.Series(price, index=idx, name="price")
print(s.head(3))
print(s.dtype, s.shape)
print(s["h2"], s.iloc[1])

h1    245.0
h2    312.0
h3    379.0
Name: price, dtype: float64
float64 (6,)
312.0 312.0


## Slide 22: Constructing a DataFrame

In [17]:
df = pd.DataFrame({"area": area, "beds": beds,
                   "price": price, "city": city},
                  index=idx)
print(df)

    area  beds  price    city
h1  1400     3  245.0  Moscow
h2  1800     3  312.0   Boise
h3  2400     4  379.0   Boise
h4  1100     2  189.0  Moscow
h5  3000     5  505.0   Boise
h6  1650     3  279.0  Moscow


## Slide 23: Reading a CSV: The Round Trip

In [18]:
df.to_csv("prices.csv", index_label="id")
back = pd.read_csv("prices.csv", index_col="id",
                   na_values=["", "NA", "?"])
print(back.dtypes.to_string())
print(back.equals(df))

area       int64
beds       int64
price    float64
city         str
True


## Slide 25: loc and iloc in Code

In [19]:
print(df.loc["h2", "price"])   # by label
print(df.iloc[1, 2])           # by position
print(df.loc["h1":"h3", ["area", "price"]])
print(df.iloc[0:3, [0, 2]])

312.0
312.0
    area  price
h1  1400  245.0
h2  1800  312.0
h3  2400  379.0
    area  price
h1  1400  245.0
h2  1800  312.0
h3  2400  379.0


## Slide 26: Pitfall: Chained Indexing

In [20]:
sub = df[df["beds"] >= 4]     # a COPY under pandas 3
sub["price"] = sub["price"] * 1.1
print(df.loc[df["beds"] >= 4, "price"].tolist())
                
df2 = df.copy()
df2.loc[df2["beds"] >= 4, "price"] *= 1.1
print(np.round(df2["price"].to_numpy(), 2))

[379.0, 505.0]
[245.  312.  416.9 189.  555.5 279. ]


## Slide 27: Check Yourself #3

In [21]:
# df has index ["h1" ... "h6"], beds = 3 3 4 2 5 3
print(df.loc["h2":"h4", "beds"].tolist())
print(df.iloc[1:4]["beds"].tolist())

[3, 4, 2]
[3, 4, 2]


## Slide 28: A Real Table: Load It

In [22]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
flowers = iris.frame.drop(columns="target")
flowers.columns = ["sepal_len", "sepal_wid",
                   "petal_len", "petal_wid"]
flowers["species"] = iris.target_names[iris.target]
print(flowers.head(3))

   sepal_len  sepal_wid  petal_len  petal_wid species
0        5.1        3.5        1.4        0.2  setosa
1        4.9        3.0        1.4        0.2  setosa
2        4.7        3.2        1.3        0.2  setosa


## Slide 29: What info() Tells You

In [23]:
flowers.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sepal_len  150 non-null    float64
 1   sepal_wid  150 non-null    float64
 2   petal_len  150 non-null    float64
 3   petal_wid  150 non-null    float64
 4   species    150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 7.2 KB


## Slide 30: describe(): Numbers at a Glance

In [24]:
print(flowers.describe().round(2))

       sepal_len  sepal_wid  petal_len  petal_wid
count     150.00     150.00     150.00     150.00
mean        5.84       3.06       3.76       1.20
std         0.83       0.44       1.77       0.76
min         4.30       2.00       1.00       0.10
25%         5.10       2.80       1.60       0.30
50%         5.80       3.00       4.35       1.30
75%         6.40       3.30       5.10       1.80
max         7.90       4.40       6.90       2.50


## Slide 31: Filtering Rows

In [25]:
wide = flowers[flowers["petal_len"] > 5.0]
print(wide.shape)
print(wide["species"].value_counts().to_string())

(42, 5)
species
virginica     41
versicolor     1


## Slide 32: End to End: One Question

In [26]:
cols = ["petal_len", "petal_wid", "species"]
tall = flowers.loc[flowers["petal_wid"] >= 1.8, cols]
print(tall.shape)
print(flowers.groupby("species")["petal_len"]
      .mean().round(2))

(46, 3)
species
setosa        1.46
versicolor    4.26
virginica     5.55
Name: petal_len, dtype: float64


## Slide 33: Check Yourself #4

In [27]:
work = flowers.copy()
sel = work[work["species"] == "setosa"]
sel["petal_len"] = 0.0
print(work["petal_len"].min())
work.loc[work["species"] == "setosa", "petal_len"] = 0.0
print(work["petal_len"].min())

1.0
0.0
